# Classification Metrics, Cross-Validation and Threshold Tuning

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    KFold, StratifiedKFold, cross_val_score, learning_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score, precision_recall_curve
)
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('All libraries loaded.')

## Exercise 1: Analyzing the Confusion Matrix

### 1.1 Definitions in the Spam Detection Context

In a binary classifier where **Positive = Spam** and **Negative = Not Spam**:

| Term | Symbol | Definition in Spam Detection |
|---|---|---|
| **True Positive (TP)** | TP | The email **is spam** and the model correctly predicted **spam**. |
| **True Negative (TN)** | TN | The email **is not spam** and the model correctly predicted **not spam**. |
| **False Positive (FP)** | FP | The email **is not spam** but the model wrongly predicted **spam** (legitimate email goes to junk). |
| **False Negative (FN)** | FN | The email **is spam** but the model wrongly predicted **not spam** (spam lands in inbox). |

In [ ]:
# ── Given confusion matrix values ────────────────────────────
TP = 90
TN = 850
FP = 30
FN = 30
total = TP + TN + FP + FN

accuracy  = (TP + TN) / total
precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1        = 2 * precision * recall / (precision + recall)

print('=== Confusion Matrix Values ===')
print(f'  TP = {TP}  |  FP = {FP}')
print(f'  FN = {FN}  |  TN = {TN}')
print(f'  Total = {total}')
print()
print('=== Computed Metrics ===')
print(f'  Accuracy  = (TP+TN)/(TP+TN+FP+FN) = ({TP}+{TN})/{total} = {accuracy:.4f}  ({accuracy*100:.2f}%)')
print(f'  Precision = TP/(TP+FP)             = {TP}/({TP}+{FP})    = {precision:.4f}  ({precision*100:.2f}%)')
print(f'  Recall    = TP/(TP+FN)             = {TP}/({TP}+{FN})    = {recall:.4f}  ({recall*100:.2f}%)')
print(f'  F1-Score  = 2*P*R/(P+R)            = {f1:.4f}  ({f1*100:.2f}%)')

In [ ]:
# ── Visual confusion matrix ───────────────────────────────────
cm_matrix = np.array([[TN, FP], [FN, TP]])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

disp = ConfusionMatrixDisplay(cm_matrix,
                               display_labels=['Not Spam', 'Spam'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix — Spam Detection', fontsize=13, fontweight='bold')

# Metrics bar chart
metrics = {'Accuracy': accuracy, 'Precision': precision,
           'Recall': recall, 'F1-Score': f1}
colors  = ['#4C72B0','#55A868','#DD8452','#C44E52']
bars = axes[1].bar(metrics.keys(), metrics.values(),
                   color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, metrics.values()):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylim(0, 1.15)
axes[1].set_title('Evaluation Metrics', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Score')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Spam Classifier — Baseline Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Scenario comparison: more FP vs more FN ──────────────────
scenarios = {
    'Baseline':    {'TP':90,  'TN':850, 'FP':30,  'FN':30},
    'High FP':     {'TP':90,  'TN':790, 'FP':90,  'FN':30},  # 3x more FP
    'High FN':     {'TP':50,  'TN':850, 'FP':30,  'FN':70},  # 3x more FN
}

rows = []
for name, v in scenarios.items():
    tot = v['TP']+v['TN']+v['FP']+v['FN']
    p   = v['TP']/(v['TP']+v['FP'])
    r   = v['TP']/(v['TP']+v['FN'])
    rows.append({
        'Scenario':  name,
        'Accuracy':  round((v['TP']+v['TN'])/tot, 3),
        'Precision': round(p, 3),
        'Recall':    round(r, 3),
        'F1-Score':  round(2*p*r/(p+r), 3),
    })

cmp_df = pd.DataFrame(rows).set_index('Scenario')
print(cmp_df.to_string())

cmp_df.plot(kind='bar', figsize=(11, 5), color=['#4C72B0','#55A868','#DD8452','#C44E52'],
            edgecolor='white', rot=0)
plt.ylim(0, 1.15)
plt.title('Metric Comparison: Baseline vs High FP vs High FN', fontsize=13, fontweight='bold')
plt.ylabel('Score')
plt.legend(loc='lower right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 1.3 Discussion — More FP vs More FN

**Higher FP (good emails flagged as spam):**  
- Precision drops significantly — many spam predictions are wrong.  
- Recall stays the same — we still catch the same proportion of real spam.  
- User experience degrades: important legitimate emails are wrongly junked, causing users to mistrust the filter.

**Higher FN (spam not caught):**  
- Recall drops significantly — a large fraction of actual spam slips through to the inbox.  
- Precision stays the same — when we do flag spam, we are still accurate.  
- Security degrades: phishing and malicious emails reach users.

**Trade-off conclusion:** In spam detection, both errors are costly in different ways. A higher FP rate annoys users; a higher FN rate exposes them to risk. Most email providers prefer to err on the side of more FNs (let some spam through) rather than FPs (never block a legitimate email), which is why precision is often prioritised.

## Exercise 2: Evaluating Trade-offs in Metrics

In [ ]:
# ── Visual illustration of Precision vs Recall trade-off ──────
thresholds = np.linspace(0.01, 0.99, 200)

# Simulate a medical classifier probability output
np.random.seed(42)
n_pos, n_neg = 200, 800
proba_pos = np.random.beta(6, 3, n_pos)   # true positives tend toward high probability
proba_neg = np.random.beta(2, 5, n_neg)   # true negatives tend toward low probability
y_true_med  = np.array([1]*n_pos + [0]*n_neg)
y_proba_med = np.concatenate([proba_pos, proba_neg])

precisions, recalls, f1s = [], [], []
for t in thresholds:
    yp = (y_proba_med >= t).astype(int)
    precisions.append(precision_score(y_true_med, yp, zero_division=0))
    recalls.append(recall_score(y_true_med, yp, zero_division=0))
    f1s.append(f1_score(y_true_med, yp, zero_division=0))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, precisions, color='#4C72B0', linewidth=2, label='Precision')
ax.plot(thresholds, recalls,    color='#E84040', linewidth=2, label='Recall')
ax.plot(thresholds, f1s,        color='#55A868', linewidth=2, label='F1-Score')
ax.axvline(0.5, color='grey', linestyle='--', linewidth=1.2, label='Default threshold (0.5)')
ax.fill_between(thresholds, precisions, recalls, alpha=0.07, color='purple')
ax.set_title('Precision / Recall / F1 vs Classification Threshold (Medical Diagnosis)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.1 Why Recall is More Important than Precision in Medical Diagnosis

In a disease detection scenario, a **False Negative** (predicting a sick patient as healthy) means the patient receives no treatment. This can have severe or fatal consequences — the disease progresses undetected.

A **False Positive** (predicting a healthy patient as sick) triggers further diagnostic tests. This causes anxiety and unnecessary cost, but the follow-up process will eventually clarify the mistake.

Therefore:
- **Recall** = how many actual sick patients we detect → must be maximised (catching every real case is critical).
- **Precision** = how accurate our positive predictions are → secondary concern (some false alarms are acceptable).

A good medical screening model should be **high recall** even at the cost of lower precision.

### 2.2 When Precision Becomes More Important than Recall

Consider a model that flags social media posts for **automated removal** for violating content policy. Here:
- A **False Positive** (legitimate post removed) directly censors a user's speech and damages platform trust.
- A **False Negative** (harmful post not caught) is handled by escalation to human reviewers.

In this scenario, **high precision** is critical — every removal decision should be very confident — while moderate recall is acceptable because human moderation forms a safety net.

### 2.3 Consequences of Focusing Solely on Accuracy in Imbalanced Datasets

Consider a dataset where 98% of emails are legitimate and 2% are spam. A model that **always predicts "not spam"** achieves:
- **Accuracy = 98%** — looks excellent on paper.
- **Recall = 0%** — catches zero spam emails.
- **F1-Score ≈ 0%** — effectively useless.

Accuracy is misleading because it is dominated by the majority class. The model appears to be performing well while completely failing at its actual purpose. Metrics like **F1-score**, **Precision**, **Recall**, and **ROC-AUC** are far more informative for imbalanced problems.

## Exercise 3: Cross-Validation and Learning Curves

In [ ]:
# ── Illustrate K-Fold vs Stratified K-Fold ───────────────────
from sklearn.datasets import make_regression

X_house, y_house = make_regression(n_samples=500, n_features=10,
                                   noise=20, random_state=42)
model_rf = RandomForestClassifier(n_estimators=50, random_state=42)

# For classification (to show stratified advantage)
X_cls, y_cls = make_classification(n_samples=500, n_features=10,
                                   weights=[0.85, 0.15], random_state=42)

kf   = KFold(n_splits=5, shuffle=True, random_state=42)
skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('=== Class ratio per fold: KFold vs StratifiedKFold ===')
print(f'{"Fold":<6} {"KFold % positive":<22} {"StratKFold % positive"}')
for i, ((_, test_kf), (_, test_skf)) in enumerate(
    zip(kf.split(X_cls, y_cls), skf.split(X_cls, y_cls)), 1
):
    pct_kf  = y_cls[test_kf].mean() * 100
    pct_skf = y_cls[test_skf].mean() * 100
    print(f'  {i:<5} {pct_kf:<22.1f} {pct_skf:.1f}')

In [ ]:
# ── Learning curves ──────────────────────────────────────────
from sklearn.linear_model import Ridge

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (title, estimator, X_lc, y_lc, scoring) in zip(axes, [
    ('Underfitting — Low-complexity model on complex data',
     Ridge(alpha=1e6),                 # very high regularisation → underfit
     X_house, y_house, 'r2'),
    ('Overfitting — High-complexity model on small data',
     RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42),
     X_cls, y_cls, 'f1'),
]):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X_lc, y_lc,
        cv=5, scoring=scoring,
        train_sizes=np.linspace(0.1, 1.0, 10),
        n_jobs=-1
    )
    tr_mean = train_scores.mean(axis=1)
    tr_std  = train_scores.std(axis=1)
    vl_mean = val_scores.mean(axis=1)
    vl_std  = val_scores.std(axis=1)

    ax.plot(train_sizes, tr_mean, 'o-', color='#4C72B0', linewidth=2, label='Training score')
    ax.plot(train_sizes, vl_mean, 's-', color='#E84040', linewidth=2, label='Validation score')
    ax.fill_between(train_sizes, tr_mean-tr_std, tr_mean+tr_std, alpha=0.12, color='#4C72B0')
    ax.fill_between(train_sizes, vl_mean-vl_std, vl_mean+vl_std, alpha=0.12, color='#E84040')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel(f'Score ({scoring})')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Learning Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.1 K-Fold vs Stratified K-Fold

| Feature | K-Fold | Stratified K-Fold |
|---|---|---|
| Split method | Randomly divides data into K equal folds | Ensures each fold has approximately the same class proportion as the full dataset |
| Suitable for | Regression / balanced classification | Imbalanced classification |
| Risk | A fold might contain very few minority-class examples | Controlled — minority class is proportionally represented in every fold |

**For housing price prediction (regression):** Standard **K-Fold** is appropriate since there is no class label to balance. The random split ensures each fold tests the model on an independent portion of the continuous target distribution.  
**For classification with imbalanced classes:** **Stratified K-Fold** is always preferred — as shown above, it produces consistent class ratios across all folds, leading to more reliable performance estimates.

### 3.2 What are Learning Curves?

A learning curve plots the **training score** and **validation score** against the **training set size**. By incrementally increasing the amount of training data and measuring performance at each step, it reveals:
- How model performance evolves as more data becomes available.
- Whether the model suffers from high bias (underfitting) or high variance (overfitting).
- Whether collecting more data would improve performance.

### 3.3 Underfitting and Overfitting from Learning Curves

**Underfitting (high bias):**  
Both training and validation scores are low and converge at a similar low value. Adding more data barely helps.  
*Remedies:* increase model complexity, add more features, reduce regularisation strength, try a more expressive algorithm.

**Overfitting (high variance):**  
Training score is high but validation score is significantly lower. A large gap between the two curves persists.  
*Remedies:* add more training data, increase regularisation, use dropout (neural networks), reduce model complexity, apply feature selection, use ensembles.

## Exercise 4: Impact of Class Imbalance on Model Evaluation

In [ ]:
# ── Simulate a rare-disease dataset (2% positive) ─────────────
X_rare, y_rare = make_classification(
    n_samples=10_000, n_features=10, weights=[0.98, 0.02],
    random_state=42
)
print(f'Dataset: {y_rare.sum()} positives out of {len(y_rare)} total ({y_rare.mean()*100:.1f}%)')

# Naive baseline: always predict majority class (no disease)
y_naive = np.zeros(len(y_rare), dtype=int)

print('\n=== Naive Classifier (always predict 0) ===')
print(f'  Accuracy  : {accuracy_score(y_rare, y_naive):.4f}  ← looks great, but...')
print(f'  Precision : {precision_score(y_rare, y_naive, zero_division=0):.4f}')
print(f'  Recall    : {recall_score(y_rare, y_naive):.4f}  ← catches ZERO sick patients')
print(f'  F1-Score  : {f1_score(y_rare, y_naive, zero_division=0):.4f}')

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

X_tr, X_te, y_tr, y_te = train_test_split(
    X_rare, y_rare, test_size=0.2, stratify=y_rare, random_state=42
)

scaler_r = StandardScaler()
X_tr_s   = scaler_r.fit_transform(X_tr)
X_te_s   = scaler_r.transform(X_te)

results_imb = {}

# Strategy 1: no adjustment
lr_plain = LogisticRegression(max_iter=500, random_state=42)
lr_plain.fit(X_tr_s, y_tr)
y_pred_plain = lr_plain.predict(X_te_s)
results_imb['No adjustment'] = y_pred_plain

# Strategy 2: class_weight='balanced'
lr_bal = LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)
lr_bal.fit(X_tr_s, y_tr)
results_imb['class_weight=balanced'] = lr_bal.predict(X_te_s)

# Strategy 3: SMOTE oversampling
try:
    sm = SMOTE(random_state=42)
    X_sm, y_sm = sm.fit_resample(X_tr_s, y_tr)
    lr_sm = LogisticRegression(max_iter=500, random_state=42)
    lr_sm.fit(X_sm, y_sm)
    results_imb['SMOTE oversampling'] = lr_sm.predict(X_te_s)
except ImportError:
    print('imbalanced-learn not installed; skipping SMOTE.')

print(f'\n{"Strategy":<25} {"Accuracy":>10} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-'*65)
for name, y_pred in results_imb.items():
    print(f'{name:<25} '
          f'{accuracy_score(y_te,y_pred):>10.3f} '
          f'{precision_score(y_te,y_pred,zero_division=0):>10.3f} '
          f'{recall_score(y_te,y_pred):>10.3f} '
          f'{f1_score(y_te,y_pred,zero_division=0):>10.3f}')

In [ ]:
# ── Metrics comparison chart ──────────────────────────────────
metric_fns = {
    'Accuracy':  lambda yp: accuracy_score(y_te, yp),
    'Precision': lambda yp: precision_score(y_te, yp, zero_division=0),
    'Recall':    lambda yp: recall_score(y_te, yp),
    'F1-Score':  lambda yp: f1_score(y_te, yp, zero_division=0),
}
strategies = list(results_imb.keys())
met_names  = list(metric_fns.keys())
data_plot  = {m: [fn(results_imb[s]) for s in strategies] for m, fn in metric_fns.items()}

x = np.arange(len(strategies))
w = 0.18
colors_m = ['#4C72B0','#55A868','#DD8452','#C44E52']

fig, ax = plt.subplots(figsize=(12, 5))
for i, (met, vals) in enumerate(data_plot.items()):
    ax.bar(x + i*w, vals, w, label=met, color=colors_m[i], edgecolor='white')

ax.set_xticks(x + 1.5*w)
ax.set_xticklabels(strategies, fontsize=10)
ax.set_ylim(0, 1.2)
ax.set_title('Impact of Imbalance Strategies on Evaluation Metrics',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 4.1 Why Accuracy is Misleading

With 98% of cases being negative, a classifier that **always predicts "no disease"** achieves 98% accuracy while completely failing its purpose: detecting sick patients. Accuracy reflects the majority class and masks the minority-class performance entirely.

### 4.2 Importance of Precision and Recall

- **Recall** is the priority: every undetected positive case means a sick patient is sent home without treatment.
- **Precision** matters for resource allocation: if recall is maximised at the cost of precision, too many healthy patients undergo costly follow-up procedures.
- **F1-Score** provides a single balanced metric that reflects performance on the minority class.

### 4.3 Strategies for Imbalanced Datasets

| Strategy | How it works | Benefit |
|---|---|---|
| `class_weight='balanced'` | Penalises misclassifying minority class more heavily during training | Simple, no data modification required |
| **SMOTE oversampling** | Synthetically generates new minority-class samples | Balances training distribution without losing majority data |
| **Undersampling** | Removes majority-class samples to balance the dataset | Fast but loses information |
| **Threshold tuning** | Lower the decision threshold below 0.5 to increase recall | No retraining required; uses PR curve |
| **Use F1 / AUC-PR** | Switch evaluation metric from accuracy to F1 or precision-recall AUC | Accurate picture of model performance on the rare class |

## Exercise 5: Role of Threshold Tuning in Classification Models

In [ ]:
# ── Simulate a loan default prediction model ──────────────────
X_loan, y_loan = make_classification(
    n_samples=5000, n_features=10, weights=[0.85, 0.15],
    random_state=42
)
X_ltr, X_lte, y_ltr, y_lte = train_test_split(
    X_loan, y_loan, test_size=0.25, stratify=y_loan, random_state=42
)
sc_l = StandardScaler()
lr_loan = LogisticRegression(max_iter=500, random_state=42, class_weight='balanced')
lr_loan.fit(sc_l.fit_transform(X_ltr), y_ltr)
y_proba_loan = lr_loan.predict_proba(sc_l.transform(X_lte))[:, 1]

thresholds_loan = np.arange(0.1, 0.91, 0.05)
pres, recs, f1s_l, accs = [], [], [], []
for t in thresholds_loan:
    yp = (y_proba_loan >= t).astype(int)
    pres.append(precision_score(y_lte, yp, zero_division=0))
    recs.append(recall_score(y_lte, yp))
    f1s_l.append(f1_score(y_lte, yp, zero_division=0))
    accs.append(accuracy_score(y_lte, yp))

# Compare threshold 0.5 vs 0.7 explicitly
for t in [0.3, 0.5, 0.7]:
    yp = (y_proba_loan >= t).astype(int)
    print(f'Threshold {t:.1f}:  '
          f'Precision={precision_score(y_lte,yp,zero_division=0):.3f}  '
          f'Recall={recall_score(y_lte,yp):.3f}  '
          f'F1={f1_score(y_lte,yp,zero_division=0):.3f}')

In [ ]:
# ── ROC curve + threshold sensitivity + PR curve ─────────────
fpr_l, tpr_l, roc_thresh = roc_curve(y_lte, y_proba_loan)
auc_l = roc_auc_score(y_lte, y_proba_loan)
optimal_idx = np.argmax(tpr_l - fpr_l)
optimal_thr = roc_thresh[optimal_idx]

prec_pr, rec_pr, pr_thresh = precision_recall_curve(y_lte, y_proba_loan)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ROC curve
axes[0].plot(fpr_l, tpr_l, color='blue', linewidth=2.5,
             label=f'AUC = {auc_l:.3f}')
axes[0].plot([0,1],[0,1],'k--', alpha=0.4)
axes[0].scatter(fpr_l[optimal_idx], tpr_l[optimal_idx],
                color='red', s=120, zorder=5,
                label=f'Optimal threshold ≈ {optimal_thr:.2f}')
axes[0].fill_between(fpr_l, tpr_l, alpha=0.10, color='blue')
axes[0].set_title('ROC Curve — Loan Default Model', fontsize=12, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Precision-Recall curve
axes[1].plot(rec_pr, prec_pr, color='#55A868', linewidth=2.5)
axes[1].set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].grid(True, alpha=0.3)

# Threshold sensitivity
axes[2].plot(thresholds_loan, pres, color='#4C72B0', linewidth=2, label='Precision')
axes[2].plot(thresholds_loan, recs, color='#E84040', linewidth=2, label='Recall')
axes[2].plot(thresholds_loan, f1s_l,color='#55A868', linewidth=2, label='F1-Score')
for t, ls in [(0.5,'--'),(0.7,':')]:
    axes[2].axvline(t, color='grey', linestyle=ls, linewidth=1.5, label=f'Threshold {t}')
axes[2].axvline(optimal_thr, color='red', linestyle='-.',
                linewidth=1.5, label=f'Optimal ({optimal_thr:.2f})')
axes[2].set_title('Metrics vs Classification Threshold', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Threshold')
axes[2].set_ylabel('Score')
axes[2].set_xlim(0.1, 0.9)
axes[2].set_ylim(0, 1.05)
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Threshold Tuning Analysis — Loan Default Prediction',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.1 Effect of Raising the Threshold from 0.5 to 0.7

Raising the threshold means the model only predicts **default** when it is at least 70% confident.

| Metric | Direction | Reason |
|---|---|---|
| **Precision** | Increases | Fewer, more confident positive predictions — a higher fraction of predicted defaults are real defaults |
| **Recall** | Decreases | More actual defaulters fall below the 0.7 threshold and are misclassified as non-defaulters |
| **False Positives** | Decrease | Fewer clients are incorrectly flagged |
| **False Negatives** | Increase | More real defaulters are missed |

### 5.2 Consequences of Setting the Threshold Too High or Too Low

**Threshold too high (e.g., 0.9):**  
- Very few loans are flagged as risky → most predictions are "no default".
- Recall collapses: many real defaulters are approved for loans.
- The bank faces large credit losses from defaults it failed to predict.

**Threshold too low (e.g., 0.2):**  
- Almost every loan is flagged as risky → very few loans are approved.
- Recall is high, but precision collapses: many creditworthy clients are rejected.
- The bank loses significant revenue from foregone interest and damages customer relationships.

**Optimal threshold:** Balance the cost of a missed default (FN cost) against the cost of a wrongly rejected application (FP cost). If the financial loss from a default is much greater than the opportunity cost of rejecting a good client, the bank should use a **lower threshold**.

### 5.3 How ROC Curves and AUC Help Find the Optimal Threshold

- The **ROC curve** plots TPR (recall) against FPR at every possible threshold. This allows visual identification of the threshold where the trade-off between catching defaulters and incorrectly flagging non-defaulters is most favourable.
- The **AUC** (Area Under the ROC Curve) measures the overall discriminative power of the model across all thresholds: 0.5 = random, 1.0 = perfect. It enables model-to-model comparison independent of any single threshold choice.
- The **optimal threshold** on the ROC curve is the point **closest to the top-left corner** (maximum TPR, minimum FPR), mathematically the point that maximises `TPR − FPR` (Youden's J statistic).
- The **Precision-Recall curve** is preferred over ROC when the dataset is heavily imbalanced: it focuses entirely on the minority class, making the precision/recall trade-off visible without being diluted by the large number of true negatives.